# 05 — Applications: RAG & Simple Tools (Instructor Notebook)

This instructor notebook shows how to combine embeddings + retrieval + a local model to answer questions (RAG). It's intentionally small-scale for workshops.

## 1) What is RAG? (what/why)

RAG (Retrieval-Augmented Generation) combines a retrieval step to fetch relevant documents with generation by an LLM. Why: LLMs can hallucinate; RAG grounds answers with real documents.

In [ ]:
# Small knowledge base
kb = [
    'CRISPR-Cas9 is a gene editing technology that allows precise edits to DNA.',
    'DNA sequencing determines the order of nucleotides in genomes.',
    'Gene therapy modifies genes to treat diseases.',
    'Protein folding determines protein function.'
]
print('KB size:', len(kb))

## 2) Minimal retrieval (what/why)

Use the embedding notebook or TF-IDF fallback to retrieve top documents for a query. Retrieval improves relevance of LLM responses.

In [ ]:
# Tiny retrieval using simple keyword overlap for portability
def retrieve_kb(query, k=2):
    scores = []
    qwords = set(query.lower().split())
    for doc in kb:
        score = len(qwords & set(doc.lower().split()))
        scores.append(score)
    idx = sorted(range(len(kb)), key=lambda i: -scores[i])[:k]
    return [kb[i] for i in idx]

q = 'How does CRISPR enable gene editing?'
print('Query:', q)
print('Retrieved:')
for d in retrieve_kb(q):
    print('-', d)

## 3) Generate answer (local model or fallback) — what/why

Combine retrieved docs into a prompt and call a small LLM (or skip generation if not available). This grounds model answers in retrieved text.

In [ ]:
# Compose prompt and try a small transformers generation if available
context = ' '.join(retrieve_kb(q, k=2))
prompt = f'Based on the following context: {context}

Answer succinctly: {q}'
print('Prompt:', prompt[:300], '...')
try:
    from transformers import pipeline
    gen = pipeline('text-generation', model='distilgpt2')
    out = gen(prompt, max_length=150, num_return_sequences=1)
    print('
Generated answer:')
    print(out[0]['generated_text'])
except Exception as e:
    print('
No generation available — use retrieved docs as the answer instead.')
    print('Retrieved answer (fallback):')
    print(context)

## Exercise D — Build a RAG prompt

Task: Modify the prompt template to ask the model to (1) cite which document provided which fact, and (2) give a short 2-sentence answer. Discuss how to structure prompts to improve factuality.